# Part 4 Transfer Learning: Final Model Reselection

This notebook documents the isolated `reselection_20260727_run1` workflow for the Member 4 contribution. ResNet18 and EfficientNet-B0 are trained under the same corrected data protocol, compared using validation metrics, and the selected EfficientNet-B0 checkpoint is evaluated once on the final test split.

Earlier experiment artifacts remain preserved for audit. The tables and figures used here are separately named so the reselection evidence does not overwrite the earlier outputs.

## 1. Research question and contribution

**Which ImageNet-pretrained architecture provides the strongest validation performance for 30-class tree-species classification under a controlled, reproducible transfer-learning protocol?**

The Member 4 contribution includes:

- ResNet18 and EfficientNet-B0 model adaptation;
- classifier-only training followed by partial fine-tuning;
- ImageNet normalisation and shared data integration;
- staged checkpoint loading and frozen BatchNorm handling;
- MPS-compatible training and evaluation;
- validation-based architecture selection;
- one locked final-test evaluation after selecting the model;
- tables, figures, documentation, and notebook material.

In [ ]:
import csv
import json
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

TABLE_DIR = ROOT / "report/tables"
FIGURE_DIR = ROOT / "report/figures/reselection_20260727_run1"
RUN_ROOT = ROOT / "reselection_20260727_run1"

required_files = [
    TABLE_DIR / "reselection_resnet18_experiments.csv",
    TABLE_DIR / "reselection_efficientnet_b0_experiments.csv",
    TABLE_DIR / "reselection_model_validation_comparison.csv",
    TABLE_DIR / "reselection_efficientnet_b0_final_test_metrics.csv",
    TABLE_DIR / "reselection_efficientnet_b0_test_per_class_metrics.csv",
]
for path in required_files:
    assert path.is_file(), f"Missing required result: {path}"

print(f"Project root: {ROOT}")
print("Reselection artifacts are available.")

## 2. Corrected data and preprocessing

| Split | Images |
|---|---:|
| Training | 4,734 |
| Validation | 1,022 |
| Test | 1,001 |
| **Total** | **6,757** |

The corrected Part 2 split contains 30 classes and removes duplicate leakage across splits. Both candidate architectures use the same split files, seed 42, batch size 16, weight decay $10^{-4}$, 224 x 224 RGB inputs, shared training augmentation, and deterministic validation/test preprocessing.

ImageNet channel statistics are used:

$$
\mu=[0.485,0.456,0.406],\qquad
\sigma=[0.229,0.224,0.225].
$$

## 3. Staged transfer-learning strategy

Both architectures use a two-stage procedure:

1. Replace the ImageNet classifier and train only the new 30-class head for 5 epochs at learning rate $10^{-3}$.
2. Load the best frozen-head checkpoint and fine-tune the final feature stage plus classifier for 10 epochs at learning rate $3\times10^{-5}$.

| Model | Total parameters | Fine-tuned parameters | Fine-tuned scope |
|---|---:|---:|---|
| ResNet18 | 11,191,902 | 8,409,118 | Layer4 + classifier |
| EfficientNet-B0 | 4,045,978 | 1,167,822 | Final two feature blocks + classifier |

EfficientNet-B0 has only **36.15%** as many total parameters as ResNet18.

In [ ]:
def read_csv_rows(path):
    with path.open(newline="", encoding="utf-8") as file:
        return list(csv.DictReader(file))

resnet_stages = read_csv_rows(
    TABLE_DIR / "reselection_resnet18_experiments.csv"
)
efficientnet_stages = read_csv_rows(
    TABLE_DIR / "reselection_efficientnet_b0_experiments.csv"
)

for row in [*resnet_stages, *efficientnet_stages]:
    print(
        f"{row['model']:15s} | {row['stage']:24s} | "
        f"best epoch {int(row['best_epoch']):2d} | "
        f"validation accuracy {float(row['val_accuracy']):.2%}"
    )

## 4. Fine-tuning behaviour

![Reselection fine-tuning curves](../report/figures/reselection_20260727_run1/reselection_finetuning_curves.png)

Both fine-tuned models converge to high validation accuracy. ResNet18 reaches its best validation accuracy at epoch 9, with a visible training-validation gap that suggests mild overfitting. EfficientNet-B0 also reaches its first best validation accuracy at epoch 9; its validation loss is lowest at the same epoch and rises slightly at epoch 10.

The saved checkpoint rule uses validation accuracy. When accuracy ties, the earlier saved checkpoint remains selected.

## 5. Validation-based architecture selection

![Validation model comparison](../report/figures/reselection_20260727_run1/reselection_validation_model_comparison.png)

| Model | Accuracy | Macro-F1 | Weighted-F1 | Top-5 accuracy | Parameters |
|---|---:|---:|---:|---:|---:|
| ResNet18 | 97.06% | 97.03% | 97.07% | 99.80% | 11.19M |
| EfficientNet-B0 | **97.16%** | **97.08%** | **97.15%** | **99.90%** | **4.05M** |

EfficientNet-B0 is selected before final test evaluation because it leads on all reported validation metrics and is substantially smaller. The 0.10 percentage-point accuracy difference is descriptive and is not evidence of statistical significance.

In [ ]:
comparison = read_csv_rows(
    TABLE_DIR / "reselection_model_validation_comparison.csv"
)

selected = next(
    row for row in comparison if row["selected_final_model"] == "True"
)
assert selected["model"] == "EfficientNet-B0"

for row in comparison:
    print(
        f"{row['model']:15s} | "
        f"accuracy={float(row['val_accuracy']):.4%} | "
        f"Macro-F1={float(row['val_macro_f1']):.4%} | "
        f"parameters={int(row['total_parameters']):,}"
    )

print(f"Selected final model: {selected['model']}")

## 6. Locked final test result

After model selection was fixed, the EfficientNet-B0 checkpoint was evaluated once on the 1,001-image final test split. No architecture, checkpoint, or hyperparameter was changed after observing the test result.

| Metric | Score |
|---|---:|
| Accuracy | **97.80%** |
| Macro Precision | 98.01% |
| Macro Recall | 97.97% |
| Macro-F1 | **97.94%** |
| Weighted-F1 | **97.81%** |
| Top-5 Accuracy | 99.90% |

The model correctly classifies **979 of 1,001** test images.

In [ ]:
test_metrics = read_csv_rows(
    TABLE_DIR / "reselection_efficientnet_b0_final_test_metrics.csv"
)[0]

assert test_metrics["split"] == "test"
assert test_metrics["evaluation_scope"] == "final_test"
assert test_metrics["test_set_evaluated"] == "True"

for metric in [
    "accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1",
    "top5_accuracy",
]:
    print(f"{metric:20s}: {float(test_metrics[metric]):.4%}")

## 7. Per-class test analysis

![Selected EfficientNet-B0 test F1 by species](../report/figures/reselection_20260727_run1/reselection_efficientnet_b0_test_per_class_f1.png)

The five lowest-F1 species are `ulmus_americana`, `ostrya_virginiana`, `diospyros_virginiana`, `styrax_japonica`, and `ulmus_rubra`. Even the lowest class F1 is 93.75%, indicating that errors are not concentrated in a severely failing class.

`ostrya_virginiana` and `styrax_japonica` have perfect recall but lower precision. `ulmus_rubra` has high precision but lower recall. These patterns identify useful targets for the group's confusion-matrix and explainability analysis.

In [ ]:
per_class = read_csv_rows(
    TABLE_DIR / "reselection_efficientnet_b0_test_per_class_metrics.csv"
)
lowest_five = sorted(per_class, key=lambda row: float(row["f1"]))[:5]

for row in lowest_five:
    print(
        f"{row['class_name']:26s} | "
        f"precision={float(row['precision']):.2%} | "
        f"recall={float(row['recall']):.2%} | "
        f"F1={float(row['f1']):.2%} | "
        f"support={int(row['support'])}"
    )

## 8. Interpretation and limitations

The reselection experiment supports three conclusions:

1. Partial fine-tuning is important for both architectures.
2. EfficientNet-B0 matches or slightly exceeds ResNet18 validation performance with far fewer parameters.
3. The validation-selected EfficientNet-B0 generalises strongly to the locked test split.

The experiment also has limitations:

- only one corrected split and one seed are used;
- the validation difference is small;
- no confidence interval or statistical significance test is reported;
- runtime and energy use were not measured consistently;
- laboratory and field performance are not separated;
- the final test set cannot be reused for further tuning.

## 9. Reproducibility

Run the isolated training stages in this order:

```bash
python part4_resnet18.py \
  --config reselection_20260727_run1/configs/resnet18_frozen.yaml

python part4_resnet18.py \
  --config reselection_20260727_run1/configs/resnet18_layer4.yaml

python part4_efficientnet_b0.py \
  --config reselection_20260727_run1/configs/efficientnet_b0_frozen.yaml

python part4_efficientnet_b0.py \
  --config reselection_20260727_run1/configs/efficientnet_b0_finetune.yaml
```

Validation evaluation remains the default. The final test command requires the explicit `--split test` argument and is recorded only for provenance.

Regenerate tracked reselection tables and figures without training or evaluation:

```bash
python part4_plot_reselection_results.py
```

## 10. Conclusion

Under the controlled reselection protocol, fine-tuned EfficientNet-B0 achieves **97.16% validation accuracy** and is selected ahead of ResNet18's **97.06%**. The locked EfficientNet-B0 checkpoint then achieves **97.80% final-test accuracy** and **97.94% test Macro-F1**.

EfficientNet-B0 is therefore the final Part 4 model: it provides the strongest reported validation result, excellent held-out performance, and a substantially smaller parameter footprint.